# 1 — Preprocessing

This notebook prepares the raw recordings for analysis:

1. Open the raw files and check their durations.
2. Extract the stimulus triggers (visual channel) for every recording.
3. Sanity-check the detected triggers.
4. *(Optional)* Spike-sort the recordings and export to phy for manual curation.
5. Extract the curated spikes per neuron.

**Assumptions:** written for **MEA 2** and **visual** stimuli (no holography). It
warns you if `params.MEA` is not 2.

**Spike sorting is optional.** Either run it here (Section 4, needs Docker), or sort
on your own machine and **skip Section 4** — then just make sure `params.py` points at
your phy output (the `.GUI` folder is auto-detected under the Sorting folder, or set
`phy_directory_override` in `params.py`).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from multiprocessing import cpu_count

# spikeinterface is heavy — this import can take 10-30 s the first time.
import spikeinterface.full as si
import probeinterface as pi

from utils import *  # pipeline helper functions
import params  # your experiment configuration

# This notebook is written for an MEA 2, visual-stimulus experiment.
if params.MEA != 2:
    print(
        f"\n/!\\ WARNING: params.MEA = {params.MEA}, but this notebook assumes MEA 2.\n"
        f"    Thresholds, probe and channel choices may be wrong for your rig.\n"
    )

In [ ]:
# --- Backup reminder: do these BEFORE analysing (Steps 0-2 of the backup guide) ---
backup_reminder(
    "Before you start (Steps 0-2)",
    [
        "Stimuli (.vec/.bin): back up to Labguru (small < 100 Mo), Nextcloud AND a hard drive. "
        "Put the DATE in the filenames. Shared stimuli can go in a shared Nextcloud folder.",
        "Also back up the stimulus-GENERATING code (the exact version you used) to Nextcloud + hard drive.",
        "Save the experimental LOG in Labguru (stimuli & parameters, rig/MEA, retina position, "
        "illumination, filters, date/session, animal id, experimenter, conditions, cell types...). "
        "A .txt / .pdf copy in the experiment folder is good too.",
        "Back up the MCD files (raw, un-sorted data) to your hard disk.",
        "Nextcloud path: Team Marre > your PI folder > Data.",
    ],
)

## Step 1 — Open the recordings and check their durations

In [ ]:
recording_names = [rec.replace(".raw", "") for rec in params.recording_names]

# Durations are inferred from the raw file sizes; check they match the names.
onsets = recording_onsets(recording_names, path=params.recording_directory)
rec_it = recording_names + ["end"]
print("Recording durations (check they are consistent with the names):\n")
for i in range(len(rec_it) - 1):
    minutes = int((onsets[rec_it[i + 1]] - onsets[rec_it[i]]) / params.fs / 60)
    print(f"  {i:>2} : {rec_it[i]:<45} {minutes} min")

## Step 2 — Extract the triggers (visual channel)

For each recording, detect the stimulus onsets on the visual trigger channel and save
them. Set `select_rec` to restrict to some recordings, or leave it empty for all.

In [ ]:
# Optionally restrict to some recordings (0-based indices); [] = all.
select_rec = []

for rec in range(len(recording_names)):
    if select_rec and rec not in select_rec:
        continue
    name = recording_names[rec]
    print(f"\n----- Triggers {rec + 1}/{len(recording_names)}  ({name}) -----")

    input_file = os.path.join(params.recording_directory, name + ".raw")
    trigger_file = os.path.join(
        params.triggers_directory, f"{params.exp}_{name}_triggers.pkl"
    )
    data_file = os.path.join(
        params.triggers_directory, f"{params.exp}_{name}_triggers_data.pkl"
    )

    if os.path.exists(data_file):
        if input("Triggers already extracted. Type Y to overwrite: ") != "Y":
            continue

    # Visual stimulus -> triggers are on the visual channel (no holography here).
    trigger_type = "visual"
    data, t_tot = load_data(input_file, channel_id=params.visual_channel_id)
    indices = detect_onsets(data, params.threshold)
    indices_errors = run_minimal_sanity_check(indices, stim_type=trigger_type)

    save_obj(
        {
            "indices": indices,
            "duration": t_tot,
            "trigger_type": trigger_type,
            "indice_errors": indices_errors,
        },
        trigger_file,
    )
    save_obj(data, data_file)
    print(f"  {len(indices)} triggers saved.")

## Step 3 — Sanity-check the triggers (high-definition, zoomable)

This opens the trigger signal in a **separate interactive window** so you can zoom in as
much as you like (use the magnifier button in the window's toolbar). The signal is drawn
at **full resolution** (no down-sampling), so zooming reveals every detail.

- Orange dots = detected triggers; red **x** = sanity-check errors; green line = threshold.
- "Detected trigger indices" should be a straight diagonal; "inter-trigger jitter" should
  stay near zero.

*Notes:* a very long recording can be slow to pan — set `plotting_range` to a
`(start_sample, end_sample)` window to focus on part of it. To return to normal inline
figures afterwards, run a cell containing `%matplotlib inline`.

In [ ]:
# Open the plot in an interactive Qt window so you can zoom freely (full resolution).
%matplotlib qt

save = False  # also save a static high-res copy to output_directory
plotting_range = False  # (start_sample, end_sample) to focus on part of the recording, or False for all

print(*[f"{i} : {name}" for i, name in enumerate(recording_names)], sep="\n")
selected = [
    int(r) for r in input("\nSelect recording(s) to check (space-separated): ").split()
]

plt.close("all")
for rec in selected:
    name = recording_names[rec]
    data = np.array(
        load_obj(
            os.path.join(
                params.triggers_directory, f"{params.exp}_{name}_triggers_data.pkl"
            )
        )
    )
    extracted = load_obj(
        os.path.join(params.triggers_directory, f"{params.exp}_{name}_triggers.pkl")
    )
    indices, err = extracted["indices"], extracted["indice_errors"]

    if plotting_range:
        lo, hi = plotting_range
        keep = (np.arange(len(data)) >= lo) & (np.arange(len(data)) < hi)
        data = data[keep]
        indices = indices[(indices >= lo) & (indices < hi)] - lo
        err = err[(err >= lo) & (err < hi)] - lo

    t = np.arange(len(data)) / params.fs  # time axis (s)

    fig = plt.figure(f"Trigger check - {name}", figsize=(14, 8))
    plt.subplot(2, 1, 1)
    plt.title(f"{name} - visual channel  (use the toolbar magnifier to zoom)")
    plt.plot(t, data, lw=0.5)  # full resolution -> zoom shows every detail
    plt.plot(
        indices / params.fs,
        data[indices],
        ".",
        markersize=3,
        color="tab:orange",
        zorder=10,
    )
    plt.axhline(params.threshold, color="green")
    plt.scatter(err / params.fs, data[err], color="red", marker="x", zorder=15)
    plt.xlabel("Time (s)")

    plt.subplot(2, 2, 3)
    plt.plot(indices)
    plt.title("Detected trigger indices")
    plt.subplot(2, 2, 4)
    plt.plot(np.diff(np.diff(indices)))
    plt.title("Inter-trigger jitter")

    plt.tight_layout()
    if save:
        fig.savefig(
            os.path.join(params.output_directory, f"{name}_triggers.png"), dpi=200
        )
    plt.show()

## Step 4 — Spike sorting *(optional)*

**Two options:**

- **Sort here** — run the cells below: filter the recordings, concatenate them, run
  spyking-circus (via Docker), and export to phy for manual curation. *Requires Docker.* (Not tested yet)
- **Sort elsewhere / already done** — **skip this whole section.** Just make sure
  `params.py` points at your phy output (the `.GUI` folder is auto-detected under the
  Sorting folder, or set `phy_directory_override` in `params.py`).

Then curate manually in phy and continue with **Step 5**.

### 4a — Filter the recordings and attach the probe

In [ ]:
# MEA probe (256 channels) used for sorting.
probe = pi.read_prb(os.path.join("./RessourcesAndTools", "mea_256_30-8iR-ITO.prb"))

recordings = {}
for name in tqdm(recording_names, desc="Filtering"):
    raw = si.read_binary(
        os.path.join(params.recording_directory, name + ".raw"),
        sampling_frequency=params.fs,
        num_channels=params.nb_channels,
        dtype="uint16",
    )
    raw = raw.set_probegroup(probe)
    filtered = si.bandpass_filter(raw, dtype="float32")
    filtered = si.common_reference(filtered)  # remove the common median
    recordings[name] = filtered
print("Filtered + median-removed all recordings.")

### 4b — Select the recordings to sort and concatenate them

In [ ]:
# Recordings to EXCLUDE from the sorting (0-based indices). [] = sort all.
skip_recording = []

recording_list = []
for rec_idx, name in enumerate(recording_names):
    if rec_idx in skip_recording:
        print(f"{rec_idx} : {name}  --> SKIPPED")
        continue
    print(f"{rec_idx} : {name}")
    recording_list.append(recordings[name])

multirecording = si.concatenate_recordings(recording_list)
print("\nConcatenated recording for sorting:")
print(multirecording)

### 4c — Run the sorter (spyking-circus, via Docker)

In [ ]:
# Spyking-circus in a Docker image (requires Docker installed and running).
custom_params = si.get_default_sorter_params("spykingcircus")
custom_params["num_workers"] = max(1, int(cpu_count() / 2) - 2)
custom_params["filter"] = False  # already filtered in 4a

sorting = si.run_sorter(
    "spykingcircus",
    recording=multirecording,
    output_folder=os.path.join(params.sorting_directory, "spyking_circus"),
    docker_image=True,
    verbose=True,
    **custom_params,
)
print(sorting)

### 4d — Export to phy

In [ ]:
nb_cpus = max(1, int(cpu_count() / 2))
waveforms_directory = os.path.join(params.sorting_directory, "waveforms")

we = si.extract_waveforms(
    multirecording,
    sorting,
    waveforms_directory,
    dtype="float32",
    chunk_memory="10M",
    overwrite=True,
    sparse=True,
    method="snr",
    threshold=1,
    n_jobs=nb_cpus,
)

print(f"Exporting to phy at {params.phy_directory}")
si.export_to_phy(
    we,
    params.phy_directory,
    copy_binary=True,
    compute_pc_features=False,
    compute_amplitudes=True,
    remove_if_exists=True,
    verbose=True,
    n_jobs=nb_cpus,
)

### 4e — Launch phy for manual curation

Opens the phy GUI. If it does not open here, run the printed command in a terminal.

In [ ]:
out = f"phy template-gui {params.phy_directory}/params.py"
!env QTWEBENGINE_CHROMIUM_FLAGS="--single-process" {out}

In [ ]:
# --- Backup reminder: manual sorting (Step 4 of the backup guide) ---
backup_reminder(
    "Step 4 - Manual sorting in phy",
    [
        "Do the manual curation in phy now (launched above, or run the printed command in a terminal).",
        "When phy is done and saved, continue below with Step 5 (extract the curated spikes).",
    ],
)

## Step 5 — Extract the curated spikes per neuron

**Run this after the manual curation in phy.** It reads the curated clusters and saves
one spike-time dictionary per neuron (split by recording) to
`<exp>_fullexp_neurons_data.pkl` — the input for all the analysis notebooks.

In [ ]:
rec_onsets = recording_onsets(recording_names, path=params.recording_directory)

cluster_number, good_clusters = extract_cluster_groups(params.phy_directory)
print(f"{len(good_clusters)} good clusters ({len(cluster_number)} total)\n")

print("Extracting spike times from phy...")
all_spike_times = extract_all_spike_times_from_phy(params.phy_directory)

print("Splitting spikes per recording, per neuron...")
good_data = split_spikes_by_recording(all_spike_times, good_clusters, rec_onsets)

good_data_file = os.path.join(
    params.output_directory, f"{params.exp}_fullexp_neurons_data.pkl"
)
save_obj(good_data, good_data_file)
print(f"\nSaved: {good_data_file}")

In [ ]:
# --- Backup reminder: save the Sorting folder (Step 5 of the backup guide) ---
backup_reminder(
    "Step 5 - Back up the Sorting folder",
    [
        "Check that the Sorting folder was properly created by the pipeline.",
        "Save the Sorting folder to your hard drive (this keeps the sorted data).",
    ],
)